# ============================================
# 07_matpower_case118_downloader.ipynb
# MATPOWER IEEE 118-Bus Test Case
# ============================================

## Dataset Title
MATPOWER IEEE 118-Bus Test Case

## Source URL
Official MATPOWER documentation page:  
https://matpower.org/docs/ref/matpower5.0/case118.html

Direct raw source file:  
https://raw.githubusercontent.com/MATPOWER/matpower/master/data/case118.m

## Purpose of This Dataset
This dataset is used as the real graph-structured power-system dataset in the PhD workflow.

It is especially useful for:
- graph construction
- node-edge modeling
- topology-aware learning
- graph neural network preparation
- system-level power-network representation

## Why We Need This Dataset
We need this dataset because it provides a structured power-network benchmark with clearly defined buses, generators, and branches.

According to the official MATPOWER documentation:
- `case118` is the IEEE 118-bus test case
- the file contains bus data
- the file contains branch data
- the file contains generator data
- the case is formatted in MATPOWER case format

This makes it highly suitable for:
- Notebook 7: Graph-ready dataset preparation
- Notebook 8: NEO-Graph
- Notebook 9: NEO-RL state-space enhancement
- Notebook 11: Full System Simulation

## What We Are Doing in This Notebook
In this notebook, we will:

1. Mount Google Drive
2. Create dataset folders
3. Download the real `case118.m` source file
4. Parse the MATPOWER file
5. Extract bus, branch, and generator tables
6. Convert them into CSV files
7. Save a dataset manifest
8. Create a clean ZIP bundle
9. Create a final package ZIP
10. Download that final package to the local PC

## Where This Data Will Be Used Next
This dataset will be used in:

- Notebook 8: NEO-Graph
- Notebook 9: NEO-RL
- Notebook 11: Full System Simulation

## Notes
The official MATPOWER page describes `case118` as the IEEE 118-bus test case, and the source code includes explicit blocks for `mpc.bus`, `mpc.gen`, and `mpc.branch`.

In [1]:
# ============================================
# 07_matpower_case118_downloader.ipynb
# FULL FIXED SINGLE CELL
# ============================================

import json
import shutil
import zipfile
import re
from pathlib import Path

import pandas as pd
import requests
from google.colab import drive, files

# -----------------------------
# 1. Dataset metadata
# -----------------------------
DATASET_NAME = "MATPOWER IEEE 118-Bus Test Case"
SOURCE_PAGE = "https://matpower.org/docs/ref/matpower5.0/case118.html"
DOWNLOAD_URL = "https://raw.githubusercontent.com/MATPOWER/matpower/master/data/case118.m"

print("Dataset Name :", DATASET_NAME)
print("Source Page  :", SOURCE_PAGE)
print("Download URL :", DOWNLOAD_URL)

# -----------------------------
# 2. Mount Google Drive
# -----------------------------
drive.mount('/content/drive')

# -----------------------------
# 3. Create folder structure
# -----------------------------
project_root = Path("/content/drive/MyDrive/PhD_Thesis_Datasets")
dataset_root = project_root / "07_matpower_case118"
raw_dir = dataset_root / "raw"
processed_dir = dataset_root / "processed"
bundle_dir = dataset_root / "bundle"

for folder in [project_root, dataset_root, raw_dir, processed_dir, bundle_dir]:
    folder.mkdir(parents=True, exist_ok=True)

print("\nCreated folders:")
print(" -", project_root)
print(" -", dataset_root)
print(" -", raw_dir)
print(" -", processed_dir)
print(" -", bundle_dir)

# -----------------------------
# 4. Robust download helper
# -----------------------------
def download_file(url: str, dest_path: Path, chunk_size: int = 1024 * 1024):
    headers = {"User-Agent": "Mozilla/5.0"}
    with requests.get(url, headers=headers, stream=True, timeout=300, allow_redirects=True) as r:
        r.raise_for_status()
        with open(dest_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
    return dest_path

# -----------------------------
# 5. Download source file
# -----------------------------
download_path = raw_dir / "case118.m"

if not download_path.exists():
    print("\nDownloading source file...")
    download_file(DOWNLOAD_URL, download_path)
    print("Download complete:", download_path)
else:
    print("\nUsing cached file:", download_path)

if not download_path.exists():
    raise FileNotFoundError(f"Source file not found: {download_path}")

print("Source file size:", round(download_path.stat().st_size / 1024, 2), "KB")

# -----------------------------
# 6. Read source file
# -----------------------------
text = download_path.read_text(encoding="utf-8", errors="ignore")

# -----------------------------
# 7. Parse MATPOWER blocks
# -----------------------------
def extract_block(full_text: str, block_name: str):
    pattern = rf"mpc\.{block_name}\s*=\s*\[(.*?)\];"
    match = re.search(pattern, full_text, flags=re.S)
    if not match:
        raise ValueError(f"Could not find block: {block_name}")
    return match.group(1)

def parse_numeric_block(block_text: str):
    rows = []
    for line in block_text.strip().splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith("%"):
            continue
        line = line.split("%")[0].strip()
        line = line.rstrip(";").strip()
        if not line:
            continue
        parts = re.split(r"\s+", line)
        try:
            row = [float(x) for x in parts if x]
            if row:
                rows.append(row)
        except Exception:
            pass
    return rows

bus_block = extract_block(text, "bus")
gen_block = extract_block(text, "gen")
branch_block = extract_block(text, "branch")

bus_rows = parse_numeric_block(bus_block)
gen_rows = parse_numeric_block(gen_block)
branch_rows = parse_numeric_block(branch_block)

print("\nParsed rows:")
print(" - bus    :", len(bus_rows))
print(" - gen    :", len(gen_rows))
print(" - branch :", len(branch_rows))

# -----------------------------
# 8. Convert to DataFrames
# -----------------------------
bus_cols = [
    "bus_i", "type", "Pd", "Qd", "Gs", "Bs", "area",
    "Vm", "Va", "baseKV", "zone", "Vmax", "Vmin"
]

gen_cols = [
    "bus", "Pg", "Qg", "Qmax", "Qmin", "Vg", "mBase",
    "status", "Pmax", "Pmin", "Pc1", "Pc2", "Qc1min",
    "Qc1max", "Qc2min", "Qc2max", "ramp_agc", "ramp_10",
    "ramp_30", "ramp_q", "apf"
]

branch_cols = [
    "fbus", "tbus", "r", "x", "b", "rateA", "rateB", "rateC",
    "ratio", "angle", "status", "angmin", "angmax"
]

bus_df = pd.DataFrame(bus_rows, columns=bus_cols[:len(bus_rows[0])])
gen_df = pd.DataFrame(gen_rows, columns=gen_cols[:len(gen_rows[0])])
branch_df = pd.DataFrame(branch_rows, columns=branch_cols[:len(branch_rows[0])])

# -----------------------------
# 9. Save CSV files
# -----------------------------
bus_csv = processed_dir / "case118_bus.csv"
gen_csv = processed_dir / "case118_gen.csv"
branch_csv = processed_dir / "case118_branch.csv"

bus_df.to_csv(bus_csv, index=False)
gen_df.to_csv(gen_csv, index=False)
branch_df.to_csv(branch_csv, index=False)

print("\nSaved processed CSV files:")
print(" -", bus_csv)
print(" -", gen_csv)
print(" -", branch_csv)

# -----------------------------
# 10. Save edge-list and node summary
# -----------------------------
edges_csv = processed_dir / "case118_edges.csv"
nodes_csv = processed_dir / "case118_nodes.csv"

edges_df = branch_df[["fbus", "tbus", "r", "x", "b", "status"]].copy()
nodes_df = bus_df[["bus_i", "type", "Pd", "Qd", "Vm", "Va", "baseKV"]].copy()

edges_df.to_csv(edges_csv, index=False)
nodes_df.to_csv(nodes_csv, index=False)

print(" -", edges_csv)
print(" -", nodes_csv)

# -----------------------------
# 11. Save manifest
# -----------------------------
manifest = {
    "dataset_name": DATASET_NAME,
    "source_page": SOURCE_PAGE,
    "download_url": DOWNLOAD_URL,
    "source_file": str(download_path),
    "processed_files": [
        str(bus_csv),
        str(gen_csv),
        str(branch_csv),
        str(edges_csv),
        str(nodes_csv),
    ],
    "counts": {
        "bus_rows": len(bus_rows),
        "gen_rows": len(gen_rows),
        "branch_rows": len(branch_rows),
    },
    "used_in_next_notebooks": [
        "Notebook 8: NEO-Graph",
        "Notebook 9: NEO-RL",
        "Notebook 11: Full System Simulation"
    ]
}

manifest_path = dataset_root / "dataset_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("\nSaved manifest:", manifest_path)

# -----------------------------
# 12. Create clean bundle ZIP
# -----------------------------
clean_bundle_zip_path = bundle_dir / "matpower_case118_CLEAN_bundle.zip"

bundle_files = [download_path, bus_csv, gen_csv, branch_csv, edges_csv, nodes_csv, manifest_path]

with zipfile.ZipFile(clean_bundle_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fp in bundle_files:
        if fp == download_path:
            arcname = "raw/" + fp.name
        elif fp == manifest_path:
            arcname = "manifest/" + fp.name
        else:
            arcname = "processed/" + fp.name
        zf.write(fp, arcname=arcname)

print("Created clean bundle ZIP:", clean_bundle_zip_path)
print("Clean bundle size:", round(clean_bundle_zip_path.stat().st_size / 1024, 2), "KB")

# -----------------------------
# 13. Create full package ZIP in Google Drive
# -----------------------------
full_zip_path = dataset_root / "matpower_case118_FULL_PACKAGE.zip"

with zipfile.ZipFile(full_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(download_path, arcname="raw/" + download_path.name)

    for fp in [bus_csv, gen_csv, branch_csv, edges_csv, nodes_csv]:
        zf.write(fp, arcname="processed/" + fp.name)

    zf.write(clean_bundle_zip_path, arcname="bundle/" + clean_bundle_zip_path.name)
    zf.write(manifest_path, arcname="manifest/" + manifest_path.name)

print("Created full package ZIP in Google Drive:", full_zip_path)
print("Full package size:", round(full_zip_path.stat().st_size / 1024, 2), "KB")

# -----------------------------
# 14. Copy final ZIP to /content for reliable PC download
# -----------------------------
runtime_copy = Path("/content/matpower_case118_FULL_PACKAGE.zip")
shutil.copy2(full_zip_path, runtime_copy)

print("Runtime backup created:", runtime_copy)
print("Runtime backup size:", round(runtime_copy.stat().st_size / 1024, 2), "KB")

# -----------------------------
# 15. Final summary
# -----------------------------
print("\n===== SUMMARY =====")
print("Dataset root         :", dataset_root)
print("Raw source file      :", download_path)
print("Processed folder     :", processed_dir)
print("Manifest             :", manifest_path)
print("Clean bundle ZIP     :", clean_bundle_zip_path)
print("Full package ZIP     :", full_zip_path)
print("Runtime backup ZIP   :", runtime_copy)
print("===================")

# -----------------------------
# 16. Download final package to PC
# -----------------------------
print("\nStarting browser download...")
files.download(str(runtime_copy))

Dataset Name : MATPOWER IEEE 118-Bus Test Case
Source Page  : https://matpower.org/docs/ref/matpower5.0/case118.html
Download URL : https://raw.githubusercontent.com/MATPOWER/matpower/master/data/case118.m
Mounted at /content/drive

Created folders:
 - /content/drive/MyDrive/PhD_Thesis_Datasets
 - /content/drive/MyDrive/PhD_Thesis_Datasets/07_matpower_case118
 - /content/drive/MyDrive/PhD_Thesis_Datasets/07_matpower_case118/raw
 - /content/drive/MyDrive/PhD_Thesis_Datasets/07_matpower_case118/processed
 - /content/drive/MyDrive/PhD_Thesis_Datasets/07_matpower_case118/bundle

Download complete: /content/drive/MyDrive/PhD_Thesis_Datasets/07_matpower_case118/raw/case118.m
Source file size: 32.91 KB

Parsed rows:
 - bus    : 118
 - gen    : 54
 - branch : 186

Saved processed CSV files:
 - /content/drive/MyDrive/PhD_Thesis_Datasets/07_matpower_case118/processed/case118_bus.csv
 - /content/drive/MyDrive/PhD_Thesis_Datasets/07_matpower_case118/processed/case118_gen.csv
 - /content/drive/MyDr

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>